# PointNet Binary Classification: Broken vs Complete 3D Objects

This notebook trains a PointNet model for binary classification on the Fantastic Breaks dataset.
- **Label 0**: Complete object (`model_c.ply`)
- **Label 1**: Broken object (`model_b_0.ply`)

**Prerequisites**: Run `scripts/prepare_classification_data.py` first to generate HDF5 data.

```
source /storage/student6/anaconda3/bin/activate pointnet
python scripts/prepare_classification_data.py \
    --data_root data/Fantastic_Breaks_v1 \
    --output_dir data/classification \
    --num_points 2048
```

## 1. Setup & Imports

In [ ]:
import os, sys, math, time
import numpy as np
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

# ── TF2 with compat.v1 for PointNet ──
import tensorflow as tf
tf1 = tf.compat.v1
tf1.disable_eager_execution()

# ── Path setup: import utilities from pointnet-master ──
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
POINTNET_DIR = os.path.join(PROJECT_ROOT, 'src', 'pointnet-master')
sys.path.insert(0, POINTNET_DIR)
sys.path.insert(0, os.path.join(POINTNET_DIR, 'models'))
sys.path.insert(0, os.path.join(POINTNET_DIR, 'utils'))

# ── Patch TF for pointnet-master TF1 compatibility ──
tf.contrib = type(sys)('contrib')
tf.contrib.layers = type(sys)('layers')
tf.contrib.layers.xavier_initializer = tf1.initializers.glorot_uniform
tf.get_variable = tf1.get_variable
tf.variable_scope = tf1.variable_scope
tf.placeholder = tf1.placeholder
tf.train = tf1.train
tf.nn.max_pool = tf1.nn.max_pool
tf.global_variables_initializer = tf1.global_variables_initializer
tf.Session = tf1.Session
tf.ConfigProto = tf1.ConfigProto
tf.summary = tf1.summary
tf.to_int64 = lambda x: tf.cast(x, tf.int64)
tf.device = tf1.device
tf.cond = tf1.cond

import tf_util
from transform_nets import input_transform_net, feature_transform_net

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

## 2. Configuration

In [ ]:
# ── Hyperparameters (adjust as needed) ──
DATA_DIR      = 'data/classification'        # HDF5 data directory
LOG_DIR       = 'results/pointnet_cls'       # output directory
GPU_INDEX     = 0                            # GPU to use
NUM_POINT     = None                         # None = use ALL points from HDF5 data
NUM_CLASSES   = 2                            # 0=complete, 1=broken
BATCH_SIZE    = 16
MAX_EPOCH     = 200
LEARNING_RATE = 0.001
MOMENTUM      = 0.9
DECAY_STEP    = 200000
DECAY_RATE    = 0.7

os.makedirs(LOG_DIR, exist_ok=True)

## 3. Data Loading & Exploration

In [ ]:
def load_h5(path):
    """Load HDF5 file with 'data' and 'label' datasets."""
    with h5py.File(path, 'r') as f:
        data = f['data'][:]
        label = f['label'][:].flatten()
    return data, label

# Load train and test sets
train_data, train_label = load_h5(os.path.join(DATA_DIR, 'train_data.h5'))
test_data, test_label   = load_h5(os.path.join(DATA_DIR, 'test_data.h5'))

# If NUM_POINT is None, use all points available in the HDF5 data
if NUM_POINT is None:
    NUM_POINT = train_data.shape[1]
    print(f"Using ALL {NUM_POINT} points per cloud (from HDF5 data shape)")
else:
    print(f"Using {NUM_POINT} points per cloud")

print(f"Train: {train_data.shape} — complete: {(train_label==0).sum()}, broken: {(train_label==1).sum()}")
print(f"Test:  {test_data.shape}  — complete: {(test_label==0).sum()}, broken: {(test_label==1).sum()}")

### Visualize Sample Point Clouds

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), subplot_kw={'projection': '3d'})
label_names = ['Complete', 'Broken']
for i, ax in enumerate(axes):
    idx = np.where(train_label == (i % 2))[0][i // 2]
    pts = train_data[idx]
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=0.5, alpha=0.6)
    ax.set_title(f'{label_names[train_label[idx]]} (idx={idx})')
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'sample_point_clouds.png'), dpi=150)
plt.show()

## 4. Data Augmentation

In [ ]:
def shuffle_data(data, labels):
    """Randomly shuffle data and labels together."""
    idx = np.arange(len(labels))
    np.random.shuffle(idx)
    return data[idx], labels[idx]

def rotate_point_cloud(batch_data):
    """Random rotation around Y-axis for each sample in the batch."""
    rotated = np.zeros(batch_data.shape, dtype=np.float32)
    for k in range(batch_data.shape[0]):
        angle = np.random.uniform() * 2 * np.pi
        c, s = np.cos(angle), np.sin(angle)
        R = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
        rotated[k] = batch_data[k] @ R
    return rotated

def jitter_point_cloud(batch_data, sigma=0.01, clip=0.05):
    """Add small Gaussian noise to each point."""
    noise = np.clip(sigma * np.random.randn(*batch_data.shape), -clip, clip)
    return batch_data + noise

print("Augmentation functions defined.")

## 5. PointNet Model Architecture (2-Class)

The PointNet model processes raw point clouds through:
1. **Input transform**: T-Net learns 3×3 rotation alignment
2. **MLP (64→64)**: Per-point feature extraction
3. **Feature transform**: T-Net learns 64×64 feature alignment
4. **MLP (64→128→1024)**: Higher-level per-point features
5. **Max pooling**: Symmetric function for permutation invariance
6. **FC (512→256→2)**: Classification head with dropout

In [ ]:
def get_model(point_cloud, is_training, bn_decay=None):
    """PointNet classification. Input: BxN; Output: Bx2."""
    batch_size = point_cloud.get_shape()[0].value
    num_point = point_cloud.get_shape()[1].value
    end_points = {}

    # Input transform (3x3)
    with tf1.variable_scope('transform_net1'):
        transform = input_transform_net(point_cloud, is_training, bn_decay, K=3)
    point_cloud_transformed = tf.matmul(point_cloud, transform)
    input_image = tf.expand_dims(point_cloud_transformed, -1)

    # Shared MLP: 64 → 64
    net = tf_util.conv2d(input_image, 64, [1,3], padding='VALID', stride=[1,1],
                         bn=True, is_training=is_training, scope='conv1', bn_decay=bn_decay)
    net = tf_util.conv2d(net, 64, [1,1], padding='VALID', stride=[1,1],
                         bn=True, is_training=is_training, scope='conv2', bn_decay=bn_decay)

    # Feature transform (64x64)
    with tf1.variable_scope('transform_net2'):
        transform = feature_transform_net(net, is_training, bn_decay, K=64)
    end_points['transform'] = transform
    net_transformed = tf.matmul(tf.squeeze(net, axis=[2]), transform)
    net_transformed = tf.expand_dims(net_transformed, [2])

    # Shared MLP: 64 → 128 → 1024
    net = tf_util.conv2d(net_transformed, 64, [1,1], padding='VALID', stride=[1,1],
                         bn=True, is_training=is_training, scope='conv3', bn_decay=bn_decay)
    net = tf_util.conv2d(net, 128, [1,1], padding='VALID', stride=[1,1],
                         bn=True, is_training=is_training, scope='conv4', bn_decay=bn_decay)
    net = tf_util.conv2d(net, 1024, [1,1], padding='VALID', stride=[1,1],
                         bn=True, is_training=is_training, scope='conv5', bn_decay=bn_decay)

    # Symmetric function: global max pooling
    net = tf_util.max_pool2d(net, [num_point,1], padding='VALID', scope='maxpool')
    net = tf.reshape(net, [batch_size, -1])

    # Classification head: FC 512 → 256 → 2
    net = tf_util.fully_connected(net, 512, bn=True, is_training=is_training,
                                  scope='fc1', bn_decay=bn_decay)
    net = tf_util.dropout(net, keep_prob=0.7, is_training=is_training, scope='dp1')
    net = tf_util.fully_connected(net, 256, bn=True, is_training=is_training,
                                  scope='fc2', bn_decay=bn_decay)
    net = tf_util.dropout(net, keep_prob=0.7, is_training=is_training, scope='dp2')
    net = tf_util.fully_connected(net, NUM_CLASSES, activation_fn=None, scope='fc3')
    return net, end_points


def get_loss(pred, label, end_points, reg_weight=0.001):
    """Cross-entropy loss + orthogonal regularization on feature transform."""
    ce_loss = tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(logits=pred, labels=label))

    # Regularize feature transform to be close to orthogonal
    transform = end_points['transform']
    K = transform.get_shape()[1].value
    mat_diff = tf.matmul(transform, tf.transpose(transform, perm=[0,2,1]))
    mat_diff -= tf.constant(np.eye(K), dtype=tf.float32)
    mat_loss = tf.nn.l2_loss(mat_diff)
    return ce_loss + mat_loss * reg_weight

print("Model architecture defined.")

## 6. Build Computation Graph

In [ ]:
def get_learning_rate(batch):
    lr = tf1.train.exponential_decay(LEARNING_RATE, batch * BATCH_SIZE,
                                     DECAY_STEP, DECAY_RATE, staircase=True)
    return tf.maximum(lr, 1e-5)

def get_bn_decay(batch):
    bn_mom = tf1.train.exponential_decay(0.5, batch * BATCH_SIZE,
                                         float(DECAY_STEP), 0.5, staircase=True)
    return tf.minimum(0.99, 1 - bn_mom)

# Build the TF graph
graph = tf.Graph()
with graph.as_default():
    with tf1.device('/gpu:' + str(GPU_INDEX)):
        pointclouds_pl = tf1.placeholder(tf.float32, (BATCH_SIZE, NUM_POINT, 3))
        labels_pl = tf1.placeholder(tf.int32, (BATCH_SIZE,))
        is_training_pl = tf1.placeholder(tf.bool, ())

        batch_var = tf.Variable(0)
        bn_decay = get_bn_decay(batch_var)
        pred, end_points = get_model(pointclouds_pl, is_training_pl, bn_decay)
        loss = get_loss(pred, labels_pl, end_points)

        correct = tf.equal(tf.argmax(pred, 1), tf.cast(labels_pl, tf.int64))
        accuracy = tf.reduce_sum(tf.cast(correct, tf.float32)) / BATCH_SIZE

        lr = get_learning_rate(batch_var)
        optimizer = tf1.train.AdamOptimizer(lr)
        train_op = optimizer.minimize(loss, global_step=batch_var)
        saver = tf1.train.Saver()

print("Computation graph built.")

## 7. Training

In [ ]:
# ── Training loop ──
config = tf1.ConfigProto()
config.gpu_options.allow_growth = True
config.allow_soft_placement = True

train_losses, train_accs = [], []
test_losses, test_accs = [], []

with tf1.Session(graph=graph, config=config) as sess:
    sess.run(tf1.global_variables_initializer(), {is_training_pl: True})
    best_acc = 0.0

    for epoch in range(MAX_EPOCH):
        # ── Train ──
        data, label = shuffle_data(train_data, train_label)
        data = data[:, :NUM_POINT, :]
        n_batches = len(data) // BATCH_SIZE
        ep_loss, ep_correct, ep_seen = 0, 0, 0

        for bi in range(n_batches):
            s, e = bi * BATCH_SIZE, (bi+1) * BATCH_SIZE
            bd = jitter_point_cloud(rotate_point_cloud(data[s:e]))
            _, lv, pv = sess.run([train_op, loss, pred],
                {pointclouds_pl: bd, labels_pl: label[s:e], is_training_pl: True})
            ep_loss += lv
            ep_correct += np.sum(np.argmax(pv, 1) == label[s:e])
            ep_seen += BATCH_SIZE

        train_losses.append(ep_loss / n_batches)
        train_accs.append(ep_correct / ep_seen)

        # ── Evaluate ──
        td = test_data[:, :NUM_POINT, :]
        tl = test_label
        n_tb = len(td) // BATCH_SIZE
        te_loss, te_correct, te_seen = 0, 0, 0

        for bi in range(n_tb):
            s, e = bi * BATCH_SIZE, (bi+1) * BATCH_SIZE
            lv, pv = sess.run([loss, pred],
                {pointclouds_pl: td[s:e], labels_pl: tl[s:e], is_training_pl: False})
            te_loss += lv * BATCH_SIZE
            te_correct += np.sum(np.argmax(pv, 1) == tl[s:e])
            te_seen += BATCH_SIZE

        te_acc = te_correct / te_seen if te_seen > 0 else 0
        test_losses.append(te_loss / te_seen if te_seen > 0 else 0)
        test_accs.append(te_acc)

        if te_acc > best_acc:
            best_acc = te_acc
            saver.save(sess, os.path.join(LOG_DIR, 'best_model.ckpt'))

        if epoch % 20 == 0 or epoch == MAX_EPOCH - 1:
            print(f"Epoch {epoch:3d} | train_loss={train_losses[-1]:.4f} train_acc={train_accs[-1]:.4f}"
                  f" | test_loss={test_losses[-1]:.4f} test_acc={test_accs[-1]:.4f}")

    # Save final
    saver.save(sess, os.path.join(LOG_DIR, 'final_model.ckpt'))
    print(f"\nTraining complete. Best test accuracy: {best_acc:.4f}")

## 8. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(test_losses, label='Test Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label='Train Accuracy')
ax2.plot(test_accs, label='Test Accuracy')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.set_title('Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 9. Evaluation on Test Set

In [ ]:
# ── Collect all predictions from best model ──
all_preds, all_probs, all_labels_eval = [], [], []

with tf1.Session(graph=graph, config=config) as sess:
    saver.restore(sess, os.path.join(LOG_DIR, 'best_model.ckpt'))
    td = test_data[:, :NUM_POINT, :]
    n_tb = len(td) // BATCH_SIZE

    for bi in range(n_tb):
        s, e = bi * BATCH_SIZE, (bi+1) * BATCH_SIZE
        pv = sess.run(pred, {pointclouds_pl: td[s:e], labels_pl: test_label[s:e],
                              is_training_pl: False})
        probs = tf.nn.softmax(pv).eval()
        all_preds.extend(np.argmax(pv, 1))
        all_probs.extend(probs[:, 1])
        all_labels_eval.extend(test_label[s:e])

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_labels_eval = np.array(all_labels_eval)

print(classification_report(all_labels_eval, all_preds, target_names=['Complete', 'Broken']))

## 10. Confusion Matrix & ROC Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(all_labels_eval, all_preds)
im = ax1.imshow(cm, cmap=plt.cm.Blues)
ax1.set_xticks([0,1]); ax1.set_yticks([0,1])
ax1.set_xticklabels(['Complete','Broken']); ax1.set_yticklabels(['Complete','Broken'])
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True'); ax1.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax1.text(j, i, str(cm[i,j]), ha='center', va='center',
                 color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
fig.colorbar(im, ax=ax1)

# ROC curve
try:
    auc = roc_auc_score(all_labels_eval, all_probs)
    fpr, tpr, _ = roc_curve(all_labels_eval, all_probs)
    ax2.plot(fpr, tpr, label=f'PointNet (AUC={auc:.3f})')
    ax2.plot([0,1],[0,1],'k--',alpha=0.5)
    ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR'); ax2.set_title('ROC Curve')
    ax2.legend(); ax2.grid(True, alpha=0.3)
except: pass

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'evaluation.png'), dpi=150)
plt.show()

## 11. Single Sample Inference

In [ ]:
def predict_single(sess_or_path, point_cloud_np):
    """
    Predict on a single point cloud [N, 3].
    Pads/samples to NUM_POINT, tiles to BATCH_SIZE for TF placeholder.
    """
    # Prepare input
    pts = point_cloud_np[:NUM_POINT]
    if len(pts) < NUM_POINT:
        idx = np.random.choice(len(pts), NUM_POINT, replace=True)
        pts = pts[idx]
    # Center + scale
    pts = pts - pts.mean(axis=0)
    s = np.max(np.linalg.norm(pts, axis=1))
    if s > 0: pts = pts / s
    # Tile to batch size
    batch = np.tile(pts[np.newaxis], (BATCH_SIZE, 1, 1))
    return batch

# ── Example: predict on first test sample ──
with tf1.Session(graph=graph, config=config) as sess:
    saver.restore(sess, os.path.join(LOG_DIR, 'best_model.ckpt'))
    batch_input = predict_single(sess, test_data[0])
    pv = sess.run(pred, {pointclouds_pl: batch_input,
                          labels_pl: np.zeros(BATCH_SIZE, dtype=np.int32),
                          is_training_pl: False})
    probs = tf.nn.softmax(pv[0:1]).eval()
    pred_label = np.argmax(probs[0])
    label_names = ['Complete', 'Broken']
    print(f"Prediction: {label_names[pred_label]} (prob: {probs[0]})")
    print(f"True label: {label_names[test_label[0]]}")